In [1]:
import gc
import os
import wandb
from typing import Dict

from mlp_training.mlp import MLP

os.environ['HF_HOME'] = '/ocean/projects/cis250042p/sjain13'
os.environ["WANDB_LOG_MODEL"] = "false"

if wandb.run is not None:
    print(f"Active run '{wandb.run.name}' found. Finishing it...")
    wandb.finish()

wandb.login(key='c7d77f7080d30d032dcac5a88bc6e3ea18058724')

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /jet/home/sjain13/.netrc
wandb: Currently logged in as: shreyj (shreyj-cmu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [2]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM

In [3]:
def get_model(model_name="meta-llama/Llama-3.1-8B-Instruct"):
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=None,  # No quantization for now
        device_map="auto",
        trust_remote_code=True,
        output_hidden_states=True,
        local_files_only=True
    )

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    return model, tokenizer

In [4]:
class MySFTTrainer(SFTTrainer):
    def __init__(self, model_name, alpha, *args, **kwargs):
        # First, call the parent class's constructor
        super().__init__(*args, **kwargs)
        # Now, initialize your custom attribute
        self.frozen_mlp = None
        self.load_mlp(model_name)
        self.alpha = alpha

        # logging changes
        self._total_base_loss_sum = 0.0
        self._total_mlp_loss_sum = 0.0
        self._loss_step_count = 0

    def load_mlp(self, model_name):
        if self.frozen_mlp is None:
            print("DBG", "Loading MLP...")

        TOTAL_MLP_LAYERS = 32 if model_name == 'l3_8' else 36
        MLP_DIMS = [(2560 if model_name == 'q3_4' else 4096), 1024, 512, 1]
        self.frozen_mlp = MLP(TOTAL_MLP_LAYERS+1, MLP_DIMS)

        state = torch.load(f"mlp_training/outputs/{model_name}_mlp_mode_lin_agt.pth")
        self.frozen_mlp.load_state_dict(state)
        self.frozen_mlp.eval()
        for param in self.frozen_mlp.parameters():
            param.requires_grad = False

        self.frozen_mlp.eval()
        
    def compute_loss(self, model, inputs, num_items_in_batch=None, return_outputs=False):
        if self.frozen_mlp is None:
            self.load_mlp()
        inputs["output_hidden_states"] = True
        outputs = model(**inputs)
        base_model_loss = outputs.loss
        # print("DBG", "Loss before MLP:", loss.item())
        
        labels = inputs.get("labels")
        hidden_states = getattr(outputs, "hidden_states", None)
        scaled_mlp_loss = torch.tensor(0.0) # Default to 0

        if hidden_states is not None:
            mask = labels != -100
            total_mlp_loss = 0
            for l in range(hidden_states[0].shape[1]):
                if any(mask[:, l]):    
                    hs = torch.stack([x[:,l,:] for x in hidden_states], dim=1)#, ).transpose(0,1)
                    mlp_loss = 1 - self.frozen_mlp(hs).squeeze(-1) # mlp predicts 1 for safe, so 1-pred = 1 for unsafe
                    mlp_loss = mlp_loss * mask[:, l].float()
                    total_mlp_loss += mlp_loss.sum()

            scaled_mlp_loss = (total_mlp_loss / mask.sum())
        
        loss = base_model_loss + (self.alpha * scaled_mlp_loss)
        # NEW: Accumulate values for logging
        # We check is_in_train to avoid accumulating during evaluation
        if self.is_in_train:
            self._total_base_loss_sum += base_model_loss.detach().item()
            self._total_mlp_loss_sum += scaled_mlp_loss.detach().item()
            self._loss_step_count += 1

        return (loss, outputs) if return_outputs else loss

    def log(self, logs: Dict[str, float], *args, **kwargs) -> None:
        """
        Log metrics to W&B. The 'logs' dict comes from the Trainer
        and already contains the averaged 'loss' (our combined loss).
        """
        # Check if we have accumulated any new loss data
        if self._loss_step_count > 0:
            # Calculate the average of our custom losses
            avg_base_model_loss = self._total_base_loss_sum / self._loss_step_count
            avg_mlp_loss = self._total_mlp_loss_sum / self._loss_step_count
            
            # Add them to the logs dictionary
            logs["base_model_loss"] = avg_base_model_loss
            logs["scaled_mlp_loss"] = avg_mlp_loss
            
            # Reset the accumulators
            self._total_base_loss_sum = 0.0
            self._total_mlp_loss_sum = 0.0
            self._loss_step_count = 0

        # Call the parent's log method to handle the actual logging
        # (which will send everything in 'logs' to W&B)
        super().log(logs)

In [5]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", 
        "k_proj", 
        "v_proj", 
        "o_proj", 
        "up_proj", 
        "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [6]:
def main(model_name: str, dataset_name="training_POC/poc_train.json", alpha=[1, 100, 300, 1000], device_map="auto"):
    dataset = load_dataset('json', data_files=dataset_name, split="train")

    LLM = ({
        "q3_4": "Qwen/Qwen3-4B-Instruct-2507", 
        "q3_8": "Qwen/Qwen3-8B", 
        "l3_8": "meta-llama/Llama-3.1-8B-Instruct"
        }).get(model_name)



    for a in alpha:
        model, tokenizer = get_model(model_name=LLM)
        model = get_peft_model(model, lora_config)

        response_template = "<|start_header_id|>assistant<|end_header_id|>" if model_name == "l3_8" else "<|im_start|>assistant\n"
        data_collator = DataCollatorForCompletionOnlyLM(
            response_template=response_template, 
            tokenizer=tokenizer
        )
        
        training_args = TrainingArguments(
            num_train_epochs=1,
            per_device_train_batch_size=4,
            gradient_accumulation_steps=8,
            optim="paged_adamw_8bit", # Memory-efficient optimizer
            logging_steps=5,
            learning_rate=2e-4,
            bf16=False, # Use bfloat16 for training if your GPU supports it
            max_grad_norm=0.3,
            warmup_ratio=0.03,
            lr_scheduler_type="cosine",
            report_to="tensorboard",
            # report_to="wandb",
            logging_strategy="steps",
            save_total_limit=1,
            output_dir=f"training_POC/results_{model_name}_alpha_{a}",      # Unique output dir
            run_name=f"run_{model_name}_alpha_{a}"
        )

        def formatting_func(data):
            # Create the chat structure that the tokenizer's chat template expects
            chats = []
            for i in range(len(data['prompt'])): 
                chats.append([{"role": "user", "content": data["prompt"][i]}, {"role": "assistant", "content": data["output"][i]}])
            return tokenizer.apply_chat_template(chats, tokenize=False)
        
        trainer = MySFTTrainer(
            model_name=model_name,
            alpha=a,
            model=model,
            train_dataset=dataset,
            peft_config=lora_config,
            formatting_func=formatting_func,
            data_collator=data_collator,
            max_seq_length=1024,
            tokenizer=tokenizer,
            args=training_args,
        )

        print(f"Training... logs and model will be saved to: {training_args.output_dir}")
        trainer.train()

        print(f"Run for alpha={a} complete. Model saved to {training_args.output_dir}")
        
        # Explicitly end the W&B run
        wandb.finish()

        del model
        del trainer
        gc.collect()
        torch.cuda.empty_cache()

    print("--- All training runs complete ---")

In [7]:
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
main("q3_4", alpha=[300,1000])

The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/jet/home/sjain13/miniconda3/envs/capstone/lib/python3.9/site-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENI

DBG Loading MLP...
Training... logs and model will be saved to: training_POC/results_q3_4_alpha_300


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cpu and cuda:0! (when checking argument for argument mat1 in method wrapper_CUDA_addmm)

# v0 POC Code

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM

# 1. Load the model and tokenizer
model_name = "meta-llama/Llama-3.1-8B-Instruct"


model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=None, # No quantization for now
    device_map="auto",  # Automatically uses available GPUs
    trust_remote_code=True,
    output_hidden_states=True,
    local_files_only=True
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
# Set the padding token to be the same as the end-of-sequence token
tokenizer.pad_token = tokenizer.eos_token

# 2. Configure LoRA (PEFT)
# LoRA is a technique to drastically reduce the number of trainable parameters.
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", 
        "k_proj", 
        "v_proj", 
        "o_proj", 
        "up_proj", 
        "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
# 3. Load and prepare the dataset
# dataset_name = "training_POC/dummy_train.json"
dataset_name = "training_POC/poc_train.json"
dataset = load_dataset('json', data_files=dataset_name, split="train")

Generating train split: 0 examples [00:00, ? examples/s]

In [4]:
from mlp_training.mlp import MLP

class MySFTTrainer(SFTTrainer):
    def __init__(self, alpha, *args, **kwargs):
        # First, call the parent class's constructor
        super().__init__(*args, **kwargs)
        # Now, initialize your custom attribute
        self.frozen_mlp = None
        self.load_mlp()
        self.alpha = alpha

        # logging changes
        self._total_base_loss_sum = 0.0
        self._total_mlp_loss_sum = 0.0
        self._loss_step_count = 0

    def load_mlp(self):
        if self.frozen_mlp is None:
            print("DBG", "Loading MLP...")
            TOTAL_MLP_LAYERS, MLP_DIMS = 32, [4096, 1024, 512, 1]
            self.frozen_mlp = MLP(TOTAL_MLP_LAYERS+1, MLP_DIMS)
            self.frozen_mlp.load_state_dict(torch.load("mlp_training/outputs/l3_8_mlp_mode_lin_agt.pth"))
            self.frozen_mlp.eval()
            for param in self.frozen_mlp.parameters():
                param.requires_grad = False
            self.frozen_mlp.to(self.model.device)
            self.frozen_mlp.eval()
        
    def compute_loss(self, model, inputs, num_items_in_batch=None, return_outputs=False):
        if self.frozen_mlp is None:
            self.load_mlp()
        inputs["output_hidden_states"] = True
        outputs = model(**inputs)
        base_model_loss = outputs.loss
        # print("DBG", "Loss before MLP:", loss.item())
        
        labels = inputs.get("labels")
        hidden_states = getattr(outputs, "hidden_states", None)
        scaled_mlp_loss = torch.tensor(0.0, device=base_model_loss.device) # Default to 0

        if hidden_states is not None:
            mask = labels != -100
            total_mlp_loss = 0
            for l in range(hidden_states[0].shape[1]):
                if any(mask[:, l]):    
                    hs = torch.stack([x[:,l,:] for x in hidden_states], dim=1)#, ).transpose(0,1)
                    mlp_loss = 1 - self.frozen_mlp(hs).squeeze(-1) # mlp predicts 1 for safe, so 1-pred = 1 for unsafe
                    mlp_loss = mlp_loss * mask[:, l].float()
                    total_mlp_loss += mlp_loss.sum()

            scaled_mlp_loss = (total_mlp_loss / mask.sum())
        
        loss = base_model_loss + (self.alpha * scaled_mlp_loss)
        # NEW: Accumulate values for logging
        # We check is_in_train to avoid accumulating during evaluation
        if self.is_in_train:
            self._total_base_loss_sum += base_model_loss.detach().item()
            self._total_mlp_loss_sum += scaled_mlp_loss.detach().item()
            self._loss_step_count += 1

        return (loss, outputs) if return_outputs else loss

    def log(self, logs: Dict[str, float], *args, **kwargs) -> None:
        """
        Log metrics to W&B. The 'logs' dict comes from the Trainer
        and already contains the averaged 'loss' (our combined loss).
        """
        # Check if we have accumulated any new loss data
        if self._loss_step_count > 0:
            # Calculate the average of our custom losses
            avg_base_model_loss = self._total_base_loss_sum / self._loss_step_count
            avg_mlp_loss = self._total_mlp_loss_sum / self._loss_step_count
            
            # Add them to the logs dictionary
            logs["base_model_loss"] = avg_base_model_loss
            logs["scaled_mlp_loss"] = avg_mlp_loss
            
            # Reset the accumulators
            self._total_base_loss_sum = 0.0
            self._total_mlp_loss_sum = 0.0
            self._loss_step_count = 0

        # Call the parent's log method to handle the actual logging
        # (which will send everything in 'logs' to W&B)
        super().log(logs)


def load_base_model():
    print("Loading fresh base model...")
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16, # Use bfloat16 if possible
        device_map="auto",
        trust_remote_code=True,
    )
    return model


def formatting_func(data):
    # Create the chat structure that the tokenizer's chat template expects
    chats = []
    for i in range(len(data['prompt'])): 
        chats.append([{"role": "user", "content": data["prompt"][i]}, {"role": "assistant", "content": data["output"][i]}])
    return tokenizer.apply_chat_template(chats, tokenize=False)

# response_template = "<|im_start|>assistant\n" # Qwen 
response_template = "<|start_header_id|>assistant<|end_header_id|>"

# Pass this string to the collator
data_collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template, 
    tokenizer=tokenizer
)

In [ ]:
alpha = [1, 100, 300, 1000]

for a in alpha:
    model = load_base_model()
    
    # Apply PEFT adapters to the fresh model
    model = get_peft_model(model, lora_config)
    
    training_args = TrainingArguments(
        num_train_epochs=1,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=8,
        optim="paged_adamw_8bit", # Memory-efficient optimizer
        logging_steps=5,
        learning_rate=2e-4,
        bf16=False, # Use bfloat16 for training if your GPU supports it
        max_grad_norm=0.3,
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",
        # report_to="tensorboard",
        report_to="wandb",
        logging_strategy="steps",
        save_total_limit=1,
        output_dir=f"training_POC/results_alpha_{a}",      # Unique output dir
        run_name=f"run_l3_8_alpha_{a}"
    )

    trainer = MySFTTrainer(
        alpha=a,
        model=model,
        train_dataset=dataset,
        peft_config=lora_config,
        formatting_func=formatting_func,
        data_collator=data_collator,
        max_seq_length=1024,
        tokenizer=tokenizer,
        args=training_args,
    )

    print(f"Training... logs and model will be saved to: {training_args.output_dir}")
    trainer.train()

    print(f"Run for alpha={a} complete. Model saved to {training_args.output_dir}")
    
    # Explicitly end the W&B run
    wandb.finish()

    del model
    del trainer
    gc.collect()
    torch.cuda.empty_cache()

print("--- All training runs complete ---")

Loading fresh base model...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: ae1d8080-a867-4760-9b42-a45e5d9f0923)')' thrown while requesting HEAD https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct/resolve/main/generation_config.json
Retrying in 1s [Retry 1/5].
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current p

Map:   0%|          | 0/13725 [00:00<?, ? examples/s]

/jet/home/sjain13/miniconda3/envs/capstone/lib/python3.9/site-packages/trl/trainer/sft_trainer.py:413: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `MySFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(
Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

'(ProtocolError('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer')), '(Request ID: 695d896b-2e4d-4c3f-b537-a0ad5e218e8e)')' thrown while requesting HEAD https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct/resolve/main/generation_config.json
Retrying in 1s [Retry 1/5].
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/jet/home/sjain13/miniconda3/envs/capstone/lib/python3.9/site-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWa

DBG Loading MLP...
Training... logs and model will be saved to: training_POC/results_alpha_1000


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Step,Training Loss
5,634.933100
10,53.476400
15,6.905600
20,5.544800
25,5.357000
30,5.509000
35,5.100300
40,4.857500
45,6.037900
50,5.222000
